# Real Temporal and Memory Data Collection Design

### Subtitle: Design Specification for Early-Stage Ransomware Detection
**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Notebook 07:** Future Temporal & Memory Telemetry Pipeline Architecture  

---

### Research Context & Purpose:
- **Current State:** The preceding research pipeline (Notebooks 01–06) successfully established supervised baselines (Random Forest, XGBoost) and demonstrated sequential decision-making using Proximal Policy Optimization (PPO).
- **Core Limitation:** The existing dataset (`cleaned_dataset.csv`) is a **static/dynamic feature-summary dataset** containing post-execution aggregate counts and entropy values. It lacks physical timestamps, sequential observation windows, and dynamic volatile memory page dumps.
- **Role of PPO to Date:** PPO was evaluated in a **synthetic temporal proxy environment** using an artificial linear feature revelation schedule ($20\%, 40\%, 60\%, 80\%, 100\%$). Consequently, real physical detection latency (in milliseconds or seconds) cannot be measured on the current data.
- **Purpose of this Notebook:** Formally design the architecture, row-level schema, observation windowing strategy, memory snapshot mechanics, and quality validation checks for a future **real temporal/memory dataset**.
- **Defensive Safety Boundary:** This notebook performs **DESIGN, SCHEMA DEFINITION, AND VALIDATION INFRASTRUCTURE ONLY**. No malware is executed, downloaded, or communicated with.


## 1. Research Gap Analysis

### Why Static Feature Summaries Cannot Prove Early-Stage Detection:
1. **Loss of Temporal Sequence:** In static feature summaries, events occurring at millisecond 50 (e.g. process injection) are lumped together with events occurring at minute 5 (e.g. volume shadow copy deletion or bulk encryption). The model cannot discern whether an indicator preceded file encryption.
2. **Inability to Measure Physical Latency:** Early detection is inherently a race against malicious payload execution. Without microsecond-level timestamps, metrics such as *Time-to-Detection (TTD)* or *Detection Before First Encryption Event* are mathematically undefined.
3. **Absence of Memory Volatility:** Dynamic memory indicators (unbacked executable memory, heap spray entropy, process hollowing markers) fluctuate dynamically over time. Post-execution static dumps miss short-lived memory transients.
4. **Endpoint Telemetry Disconnect:** Production endpoint detection and response (EDR) systems process continuous event streams ($t_0, t_1, t_2, \dots$), not aggregated summary vectors. Addressing this requires a streaming temporal telemetry design.


In [1]:
import json
from pathlib import Path
import sys
from typing import Any, Dict, List, Optional, Tuple
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.utils import get_reports_dir, get_figures_dir
reports_dir = get_reports_dir()
figures_dir = get_figures_dir()

current_vs_future = pd.DataFrame([
    {
        'Dimension': 'Data Nature',
        'Current Dataset': 'Static/Dynamic Feature Summary',
        'Future Real Dataset': 'Continuous Streaming Event & Memory Log',
    },
    {
        'Dimension': 'Temporal Metadata',
        'Current Dataset': 'None (No timestamps, no sequence index)',
        'Future Real Dataset': 'High-resolution microsecond timestamps (UTC)',
    },
    {
        'Dimension': 'Event Ordering',
        'Current Dataset': 'Unordered post-mortem aggregates',
        'Future Real Dataset': 'Strictly chronological event causality',
    },
    {
        'Dimension': 'Memory Telemetry',
        'Current Dataset': 'Static overall entropy column only',
        'Future Real Dataset': 'Dynamic working set, page entropy, handle counts',
    },
    {
        'Dimension': 'Observation Unit',
        'Current Dataset': 'Single execution summary row per sample',
        'Future Real Dataset': 'Time-bucketed observation windows (W_0, W_1, ...)',
    },
    {
        'Dimension': 'Ground Truth',
        'Current Dataset': 'Family label only (G, E, L)',
        'Future Real Dataset': 'Family label + exact timestamp of malicious onset',
    },
    {
        'Dimension': 'Measurable Latency',
        'Current Dataset': 'Synthetic observation steps only (t in [0, 4])',
        'Future Real Dataset': 'True physical time-to-detection (ms/sec)',
    },
])

def df_to_markdown_table(df: pd.DataFrame) -> str:
    headers = [str(col) for col in df.columns]
    h_line = '| ' + ' | '.join(headers) + ' |'
    s_line = '| ' + ' | '.join(['---'] * len(headers)) + ' |'
    d_lines = ['| ' + ' | '.join(str(val) for val in row.values) + ' |' for _, row in df.iterrows()]
    return '\n'.join([h_line, s_line] + d_lines)

md_report_path = reports_dir / 'current_vs_future_data.md'
with open(md_report_path, 'w', encoding='utf-8') as f:
    f.write('# Comparison: Current Feature-Summary vs. Future Temporal Dataset\n\n')
    f.write(df_to_markdown_table(current_vs_future))

print('Current vs. Future Data Architecture Comparison:')
print(current_vs_future.to_string(index=False))
print(f'\nSaved comparison report to: {md_report_path}')


Current vs. Future Data Architecture Comparison:
         Dimension                                Current Dataset                               Future Real Dataset
       Data Nature                 Static/Dynamic Feature Summary           Continuous Streaming Event & Memory Log
 Temporal Metadata        None (No timestamps, no sequence index)      High-resolution microsecond timestamps (UTC)
    Event Ordering               Unordered post-mortem aggregates            Strictly chronological event causality
  Memory Telemetry             Static overall entropy column only  Dynamic working set, page entropy, handle counts
  Observation Unit        Single execution summary row per sample Time-bucketed observation windows (W_0, W_1, ...)
      Ground Truth                    Family label only (G, E, L) Family label + exact timestamp of malicious onset
Measurable Latency Synthetic observation steps only (t in [0, 4])          True physical time-to-detection (ms/sec)

Saved comparison repor

In [2]:
print('Future Real Temporal / Memory Dataset Schema Specification:')
print('=' * 70)
print('Proposed Future Fields (Design Specification - Not Yet Collected):\n')

schema_fields = [
    # Identification & Windowing
    ('sample_id', 'string', 'Unique session UUID for the sample execution run', True, False, False),
    ('process_id', 'int64', 'Operating system process ID (PID)', True, False, False),
    ('parent_process_id', 'int64', 'Parent process ID (PPID) in the process tree', True, False, False),
    ('timestamp_us', 'int64', 'Microsecond epoch timestamp of observation window close', True, True, False),
    ('relative_time_ms', 'float64', 'Elapsed execution time in milliseconds from process start', True, True, False),
    ('observation_window', 'int32', 'Sequential window index (0, 1, 2, ...)', True, True, False),
    ('process_name', 'string', 'Name of the executing binary image (e.g. svchost.exe)', True, False, False),
    ('process_path', 'string', 'Full execution path on disk', False, False, False),
    ('label', 'string', 'Binary label: benign or ransomware', True, False, False),
    ('family', 'string', 'Specific malware family or benign baseline profile', True, False, False),
    
    # Behavioral Rate Telemetry
    ('api_count', 'int32', 'Total API invocations in current window', True, True, False),
    ('api_rate', 'float64', 'API calls per millisecond in window', True, True, False),
    ('dll_count', 'int32', 'Count of newly loaded DLL modules in window', True, True, False),
    ('dll_load_rate', 'float64', 'DLL loads per second', True, True, False),
    ('registry_read_count', 'int32', 'Registry key query and open count in window', True, True, False),
    ('registry_write_count', 'int32', 'Registry value modifications and key creations in window', True, True, False),
    ('file_read_count', 'int32', 'File system read operations in window', True, True, False),
    ('file_write_count', 'int32', 'File system write/modify operations in window', True, True, False),
    ('file_create_count', 'int32', 'New files created in window', True, True, False),
    ('directory_activity_count', 'int32', 'Directory enumeration and walk operations', True, True, False),
    ('entropy', 'float64', 'Average Shannon entropy of written file buffers and memory pages', True, True, True),
    
    # Dynamic Memory Telemetry
    ('memory_snapshot_id', 'string', 'Unique identifier for the associated volatile memory dump', False, True, True),
    ('working_set_bytes', 'int64', 'Current process physical working set size in bytes', True, True, True),
    ('private_bytes', 'int64', 'Committed private memory allocation in bytes', True, True, True),
    ('handle_count', 'int32', 'Total open kernel object handles (files, keys, mutexes)', True, True, True),
    ('thread_count', 'int32', 'Number of active threads in the process', True, True, True),
    ('network_connection_count', 'int32', 'Active TCP/UDP sockets associated with PID', True, True, False),
    ('loaded_module_count', 'int32', 'Total count of DLL modules mapped in process space', True, True, True),
]

df_schema = pd.DataFrame(schema_fields, columns=[
    'Column', 'Type', 'Description', 'Required?', 'Temporal?', 'Memory-derived?'
])
print(f'Total Proposed Schema Fields: {len(df_schema)}')
print(f'Temporal Fields:            {df_schema["Temporal?"].sum()}')
print(f'Memory-Derived Fields:      {df_schema["Memory-derived?"].sum()}')


Future Real Temporal / Memory Dataset Schema Specification:
Proposed Future Fields (Design Specification - Not Yet Collected):

Total Proposed Schema Fields: 28
Temporal Fields:            21
Memory-Derived Fields:      7


In [3]:
schema_csv_path = reports_dir / 'temporal_dataset_schema.csv'
df_schema.to_csv(schema_csv_path, index=False)

print('Proposed Temporal & Memory Dataset Schema:')
print(df_schema.to_string(index=False))
print(f'\nSaved schema specification CSV to: {schema_csv_path}')


Proposed Temporal & Memory Dataset Schema:
                  Column    Type                                                      Description  Required?  Temporal?  Memory-derived?
               sample_id  string                 Unique session UUID for the sample execution run       True      False            False
              process_id   int64                                Operating system process ID (PID)       True      False            False
       parent_process_id   int64                     Parent process ID (PPID) in the process tree       True      False            False
            timestamp_us   int64          Microsecond epoch timestamp of observation window close       True       True            False
        relative_time_ms float64        Elapsed execution time in milliseconds from process start       True       True            False
      observation_window   int32                           Sequential window index (0, 1, 2, ...)       True       True            Fals

In [4]:
print('Configurable Observation Windowing Strategy:')
print('=' * 70)

# Window Configuration Parameters (Configurable Framework)
WINDOW_CONFIG = {
    'WINDOW_SIZE_MS': 100,    # Duration of each discrete observation window in ms
    'WINDOW_STEP_MS': 50,     # Sliding step stride in ms (50% overlap for sliding windows)
    'MIN_EPISODE_WINDOWS': 10, # Minimum required observations for valid episode
    'MAX_EPISODE_WINDOWS': 300, # 30 seconds maximum monitoring budget per sample
    'CANARY_ALERT_OFFSET_MS': 0, # Timestamp offset relative to canary file access
}

for k, v in WINDOW_CONFIG.items():
    print(f'  {k:<25}: {v}')

print('\nMathematical Formulation:')
print('Observation Window W_k spans time interval [k * WINDOW_STEP_MS, k * WINDOW_STEP_MS + WINDOW_SIZE_MS].')
print('Events falling within [t_start, t_end) are aggregated into window-level feature rates.')
print('Note: Final parameters must be calibrated during initial sandbox instrumentation to balance CPU overhead with temporal resolution.')


Configurable Observation Windowing Strategy:
  WINDOW_SIZE_MS           : 100
  WINDOW_STEP_MS           : 50
  MIN_EPISODE_WINDOWS      : 10
  MAX_EPISODE_WINDOWS      : 300
  CANARY_ALERT_OFFSET_MS   : 0

Mathematical Formulation:
Observation Window W_k spans time interval [k * WINDOW_STEP_MS, k * WINDOW_STEP_MS + WINDOW_SIZE_MS].
Events falling within [t_start, t_end) are aggregated into window-level feature rates.
Note: Final parameters must be calibrated during initial sandbox instrumentation to balance CPU overhead with temporal resolution.


In [5]:
print('Temporal Feature Engineering Specifications:')
print('=' * 70)

feature_engineering_specs = pd.DataFrame([
    {
        'Feature Class': 'Window Rates',
        'Derived Feature': 'api_rate, file_write_rate, reg_write_rate',
        'Formula / Definition': 'event_count / (window_duration_ms / 1000.0)',
        'Defensive Relevance': 'Sudden spike indicates automated high-speed encryption activity.',
    },
    {
        'Feature Class': 'First-Order Deltas',
        'Derived Feature': 'delta_entropy',
        'Formula / Definition': 'entropy[k] - entropy[k-1]',
        'Defensive Relevance': 'Positive delta marks transition from benign I/O to ciphertext I/O.',
    },
    {
        'Feature Class': 'Memory Deltas',
        'Derived Feature': 'delta_private_bytes, delta_working_set',
        'Formula / Definition': 'bytes[k] - bytes[k-1]',
        'Defensive Relevance': 'Buffer allocation for staging file encryption keys or payload blocks.',
    },
    {
        'Feature Class': 'Handle & Thread Deltas',
        'Derived Feature': 'delta_handles, delta_threads',
        'Formula / Definition': 'handles[k] - handles[k-1], threads[k] - threads[k-1]',
        'Defensive Relevance': 'Multi-threaded ransomware spawning worker threads for parallel encryption.',
    },
    {
        'Feature Class': 'Rolling Statistics',
        'Derived Feature': 'rolling_mean_file_writes, rolling_std_entropy',
        'Formula / Definition': 'Moving average over prior m=5 windows',
        'Defensive Relevance': 'Smooths bursty I/O noise while preserving persistent behavioral shifts.',
    },
])

print(feature_engineering_specs.to_string(index=False))


Temporal Feature Engineering Specifications:
         Feature Class                               Derived Feature                                 Formula / Definition                                                        Defensive Relevance
          Window Rates     api_rate, file_write_rate, reg_write_rate          event_count / (window_duration_ms / 1000.0)           Sudden spike indicates automated high-speed encryption activity.
    First-Order Deltas                                 delta_entropy                            entropy[k] - entropy[k-1]         Positive delta marks transition from benign I/O to ciphertext I/O.
         Memory Deltas        delta_private_bytes, delta_working_set                                bytes[k] - bytes[k-1]      Buffer allocation for staging file encryption keys or payload blocks.
Handle & Thread Deltas                  delta_handles, delta_threads handles[k] - handles[k-1], threads[k] - threads[k-1] Multi-threaded ransomware spawning worker thr

In [6]:
print('Volatile Memory Snapshot Design Architecture:')
print('=' * 70)

memory_telemetry_schema = pd.DataFrame([
    {'Metric': 'memory_snapshot_id', 'Type': 'UUID', 'Description': 'Unique identifier linking event window to full memory dump'},
    {'Metric': 'working_set_bytes', 'Type': 'int64', 'Description': 'Physical RAM pages currently assigned to the monitored process'},
    {'Metric': 'private_bytes', 'Type': 'int64', 'Description': 'Committed virtual memory pages unshared with other processes'},
    {'Metric': 'unbacked_executable_pages', 'Type': 'int32', 'Description': 'Count of executable memory regions not backed by an on-disk image (shellcode/injection)'},
    {'Metric': 'heap_entropy_mean', 'Type': 'float64', 'Description': 'Average Shannon entropy across process heap allocations'},
    {'Metric': 'stack_pivot_detected', 'Type': 'boolean', 'Description': 'Flag indicating stack pointer points outside designated thread stack boundaries'},
    {'Metric': 'mapped_in_memory_dlls', 'Type': 'int32', 'Description': 'Count of PE images in memory lacking corresponding disk lock'},
])

print(memory_telemetry_schema.to_string(index=False))
print('\nHigh-Level Instrumentation Principle:')
print('Memory features are captured via hypervisor-level Virtual Machine Introspection (VMI) or kernel-level ETW (Microsoft-Windows-Kernel-Memory).')
print('Defensive Safety Guarantee: No malware execution is conducted; telemetry specification describes passive monitoring instrumentation.')


Volatile Memory Snapshot Design Architecture:
                   Metric    Type                                                                             Description
       memory_snapshot_id    UUID                              Unique identifier linking event window to full memory dump
        working_set_bytes   int64                          Physical RAM pages currently assigned to the monitored process
            private_bytes   int64                            Committed virtual memory pages unshared with other processes
unbacked_executable_pages   int32 Count of executable memory regions not backed by an on-disk image (shellcode/injection)
        heap_entropy_mean float64                                 Average Shannon entropy across process heap allocations
     stack_pivot_detected boolean         Flag indicating stack pointer points outside designated thread stack boundaries
    mapped_in_memory_dlls   int32                            Count of PE images in memory lacking co

In [7]:
print('Process Hierarchy & Parent-Child Temporal Tracking:')
print('=' * 70)

process_tree_schema = pd.DataFrame([
    {'Field': 'process_id (PID)', 'Type': 'int64', 'Role': 'Primary identifier of the monitored process entity'},
    {'Field': 'parent_process_id (PPID)', 'Type': 'int64', 'Role': 'Identifier of spawning process; traces execution chain'},
    {'Field': 'process_name', 'Type': 'string', 'Role': 'Binary name from PE header'},
    {'Field': 'process_path', 'Type': 'string', 'Role': 'Filesystem path (detects execution from Temp, AppData)'},
    {'Field': 'process_start_time_us', 'Type': 'int64', 'Role': 'Exact epoch microsecond when NtCreateUserProcess completed'},
    {'Field': 'process_end_time_us', 'Type': 'int64', 'Role': 'Exact epoch microsecond of process termination (or null)'},
    {'Field': 'child_process_count', 'Type': 'int32', 'Role': 'Total child processes spawned up to current window'},
    {'Field': 'tree_depth', 'Type': 'int32', 'Role': 'Depth of PID in the process ancestor tree'},
])

print(process_tree_schema.to_string(index=False))
print('\nContextual Value: Malicious ransomware droppers frequently spawn helper utilities (e.g. vssadmin.exe, bcdedit.exe).')
print('Temporal tracking of process tree expansion provides critical context before file encryption commences.')


Process Hierarchy & Parent-Child Temporal Tracking:
                   Field   Type                                                       Role
        process_id (PID)  int64         Primary identifier of the monitored process entity
parent_process_id (PPID)  int64     Identifier of spawning process; traces execution chain
            process_name string                                 Binary name from PE header
            process_path string     Filesystem path (detects execution from Temp, AppData)
   process_start_time_us  int64 Exact epoch microsecond when NtCreateUserProcess completed
     process_end_time_us  int64   Exact epoch microsecond of process termination (or null)
     child_process_count  int32         Total child processes spawned up to current window
              tree_depth  int32                  Depth of PID in the process ancestor tree

Contextual Value: Malicious ransomware droppers frequently spawn helper utilities (e.g. vssadmin.exe, bcdedit.exe).
Temporal tra

In [8]:
print('Raw Event Stream Table Definition (Pre-Windowing Ingestion):')
print('=' * 70)

event_categories = pd.DataFrame([
    {'Event Category': 'API', 'Example Providers': 'Microsoft-Windows-Kernel-Process, Win32 API Hooking', 'High-Risk Signatures': 'VirtualAllocEx, WriteProcessMemory, CryptEncrypt'},
    {'Event Category': 'DLL_LOAD', 'Example Providers': 'ImageLoad ETW, LdrLoadDll', 'High-Risk Signatures': 'advapi32.dll, crypt32.dll, bcrypt.dll'},
    {'Event Category': 'REGISTRY', 'Example Providers': 'Microsoft-Windows-Kernel-Registry', 'High-Risk Signatures': 'Run keys, SafeBoot modification, Volume shadow access'},
    {'Event Category': 'FILE', 'Example Providers': 'Minifilter driver, Microsoft-Windows-Kernel-File', 'High-Risk Signatures': 'Massive rename, extension changes, canary file access'},
    {'Event Category': 'PROCESS', 'Example Providers': 'PsSetCreateProcessNotifyRoutineEx', 'High-Risk Signatures': 'Spawning cmd.exe, powershell.exe, vssadmin.exe'},
    {'Event Category': 'MEMORY', 'Example Providers': 'Hypervisor memory trap, VMI, QueryWorkingSetEx', 'High-Risk Signatures': 'Page protection RW -> RWX, rapid private byte inflation'},
])

print(event_categories.to_string(index=False))


Raw Event Stream Table Definition (Pre-Windowing Ingestion):
Event Category                                   Example Providers                                    High-Risk Signatures
           API Microsoft-Windows-Kernel-Process, Win32 API Hooking        VirtualAllocEx, WriteProcessMemory, CryptEncrypt
      DLL_LOAD                           ImageLoad ETW, LdrLoadDll                   advapi32.dll, crypt32.dll, bcrypt.dll
      REGISTRY                   Microsoft-Windows-Kernel-Registry   Run keys, SafeBoot modification, Volume shadow access
          FILE    Minifilter driver, Microsoft-Windows-Kernel-File   Massive rename, extension changes, canary file access
       PROCESS                   PsSetCreateProcessNotifyRoutineEx          Spawning cmd.exe, powershell.exe, vssadmin.exe
        MEMORY      Hypervisor memory trap, VMI, QueryWorkingSetEx Page protection RW -> RWX, rapid private byte inflation


## 2. Multi-Level Labeling Strategy

A future temporal dataset must maintain dual-level labeling to support both static supervised baselines and sequential stopping RL agents:

### 1. Sample-Level Ground Truth:
- **`sample_label`:** Binary indicator $\in \{\text{benign}, \text{ransomware}\}$.
- **`family_label`:** Granular taxonomy (e.g. historical families `G`, `E`, `L`, modern variants, or benign baseline role `office_user`, `developer_tools`).

### 2. Window-Level Ground Truth:
Each discrete observation window $W_k$ receives a temporal phase label conditioned on execution milestones:
- **`Phase 0: Benign / Baseline Activity`:** Normal OS and benign application behavior.
- **`Phase 1: Pre-Encryption Suspicious Stage`:** Reconnaissance, persistence setup, process injection, privilege escalation. (The target detection window for PPO early stopping).
- **`Phase 2: Active Encryption Stage`:** Mass modification of user documents, canary file destruction, ransom note generation.
- **`Phase 3: Post-Damage Phase`:** Wallpaper modification, process suicide, key exfiltration.


## 3. Objective Ground Truth & Milestone Synchronization

To avoid subjective labeling of early-stage behavior, future sandbox runs must record hardware-synchronized execution milestones:

```
t_0 (Process Launch)
 │
 ├── t_inj (First Inter-Process Code Injection)
 │
 ├── t_shadow (Volume Shadow Copy Tampering / bcdedit)
 │
 ├── t_canary (Access to Pre-Positioned Honeypot Canary Files)  <-- CRITICAL MILESTONE: t_onset
 │
 ├── t_first_enc (First User Document Encrypted on Filesystem)
 │
 └── t_term (Process Termination / Ransom Note Display)
```

- **Definition of Malicious Onset ($t_{\text{onset}}$):** The exact microsecond when the malicious process accesses the first monitored canary file or initiates bulk cryptographic key setup.
- **Early-Stage Detection Target:** The detector must issue `RANSOMWARE_ALERT` at some time $t_{\text{alert}} < t_{\text{first\_enc}}$. Alerts occurring after $t_{\text{first\_enc}}$ represent damage mitigation rather than early-stage prevention.


In [9]:
print('Mathematical Definitions for Future Temporal Metrics:')
print('=' * 70)

def calculate_detection_latency(t_alert_ms: float, t_onset_ms: float) -> float:
    """Physical time elapsed between malicious onset and detector intervention."""
    return t_alert_ms - t_onset_ms

def is_early_detection(t_alert_ms: float, t_first_damage_ms: float) -> bool:
    """True if detection alert was triggered prior to irreversible filesystem encryption."""
    return t_alert_ms < t_first_damage_ms

print('1. Physical Detection Latency:')
print('   Latency = t_alert - t_onset (measured in milliseconds)')
print('\n2. Early Detection Binary Criterion:')
print('   Early_Success = True IF (t_alert < t_first_damage) ELSE False')
print('\n3. Early Detection Window Allowance:')
print('   tau_early = Configurable parameter representing allowable pre-encryption inspection budget.')
print('\nSCIENTIFIC INVARIANT:')
print('These functions represent the mathematical design for future experiments. No values are calculated here because no real temporal dataset exists.')


Mathematical Definitions for Future Temporal Metrics:
1. Physical Detection Latency:
   Latency = t_alert - t_onset (measured in milliseconds)

2. Early Detection Binary Criterion:
   Early_Success = True IF (t_alert < t_first_damage) ELSE False

3. Early Detection Window Allowance:
   tau_early = Configurable parameter representing allowable pre-encryption inspection budget.

SCIENTIFIC INVARIANT:
These functions represent the mathematical design for future experiments. No values are calculated here because no real temporal dataset exists.


In [10]:
print('Future Streaming PPO State Formulation:')
print('=' * 70)

print('Transition from Synthetic Proxy to Real Streaming MDP:')
print('  Current Phase (Notebook 05):')
print('    Static Feature Vector X (16 dim) -> Multiplied by reveal schedule [0.2, 0.4, ...] -> PPO')
print('  Future Phase:')
print('    Physical Streaming Window W_k -> Extracted Feature Vector O_k -> PPO\n')

print('Future Real Observation Vector O_k in R^d consists of 4 concatenated sub-vectors:')
print('  O_k = [')
print('    f_behavioral (Window Rates: api_rate, file_rate, reg_rate, dll_rate),')
print('    f_memory     (Working Set, Private Bytes, Page Entropy, Unbacked Pages),')
print('    f_deltas     (Delta Entropy, Delta Memory, Delta Handles, Delta Threads),')
print('    f_rolling    (Moving Average Rates over past m=5 windows)')
print('  ]')
print('\nThis state space preserves the Markovian property while capturing rate of behavioral change.')


Future Streaming PPO State Formulation:
Transition from Synthetic Proxy to Real Streaming MDP:
  Current Phase (Notebook 05):
    Static Feature Vector X (16 dim) -> Multiplied by reveal schedule [0.2, 0.4, ...] -> PPO
  Future Phase:
    Physical Streaming Window W_k -> Extracted Feature Vector O_k -> PPO

Future Real Observation Vector O_k in R^d consists of 4 concatenated sub-vectors:
  O_k = [
    f_behavioral (Window Rates: api_rate, file_rate, reg_rate, dll_rate),
    f_memory     (Working Set, Private Bytes, Page Entropy, Unbacked Pages),
    f_deltas     (Delta Entropy, Delta Memory, Delta Handles, Delta Threads),
    f_rolling    (Moving Average Rates over past m=5 windows)
  ]

This state space preserves the Markovian property while capturing rate of behavioral change.


## 4. Future PPO Action Space Specification

The action space validated in `src/ppo_environment.py` is directly preserved for future streaming deployment:

$$\mathcal{A} = \{0, 1, 2\}$$

| Action Index | Action Name | Operational Semantics | System Response |
| :---: | :--- | :--- | :--- |
| **0** | `CONTINUE_MONITORING` | Activity appears benign or evidence is currently inconclusive; accumulate telemetry in next window $W_{k+1}$. | Passive logging; zero process disruption. |
| **1** | `SUSPICIOUS` | Suspicious behavioral deviation observed; elevate logging granularity or request kernel memory introspection. | Throttle I/O priority; take lightweight memory snapshot. |
| **2** | `RANSOMWARE_ALERT` | Definitive malicious ransomware behavior identified; sequential episode halts immediately. | Suspend thread/process; isolate network socket; trigger EDR quarantine. |

This discrete 3-action design directly models the operational escalation ladder of automated defensive endpoint agents.


In [11]:
print('Future Temporal Reward Formulation (Conceptual Design):')
print('=' * 70)

print('R(s_k, a_k, s_{k+1}) =')
print('  + R_alert_tp       (if a_k == ALERT and sample is ransomware at t < t_damage)')
print('  + Bonus_early(t)   (bonus scaling inversely with elapsed detection time: f(t_damage - t))')
print('  - R_false_alert    (if a_k == ALERT on a benign execution process)')
print('  - R_late_penalty   (if a_k == ALERT but after t >= t_damage)')
print('  - R_missed         (if episode terminates without alert on ransomware)')
print('  - C_monitor        (small step cost per window to penalize indefinite monitoring)')
print('\nCalibration Strategy:')
print('Reward weights must be empirically calibrated once the true base rate of benign vs. ransomware processes is established in the live environment.')


Future Temporal Reward Formulation (Conceptual Design):
R(s_k, a_k, s_{k+1}) =
  + R_alert_tp       (if a_k == ALERT and sample is ransomware at t < t_damage)
  + Bonus_early(t)   (bonus scaling inversely with elapsed detection time: f(t_damage - t))
  - R_false_alert    (if a_k == ALERT on a benign execution process)
  - R_late_penalty   (if a_k == ALERT but after t >= t_damage)
  - R_missed         (if episode terminates without alert on ransomware)
  - C_monitor        (small step cost per window to penalize indefinite monitoring)

Calibration Strategy:
Reward weights must be empirically calibrated once the true base rate of benign vs. ransomware processes is established in the live environment.


## 5. Temporal Train/Test Partitioning Protocols

To rigorously evaluate future models and prevent data leakage, future experimental pipelines must support multiple orthogonal partitioning strategies:

```
Partitioning Strategy 1: Sample-Level Stratified Split
├── Randomly partitions distinct execution runs into 80% train / 20% test.
└── Useful for validating consistency within a known set of families.

Partitioning Strategy 2: Family-Disjoint Split (Zero-Day Evaluation)
├── Families [A, B, C] assigned exclusively to Training.
└── Family [D] held out strictly for Testing.
└── Evaluates whether learned sequential policies generalize to novel ransomware variants.

Partitioning Strategy 3: Time-Based Chronological Split (Production Mirror)
├── Training: Samples collected from Year Y, Months 1–6.
└── Testing:  Samples collected from Year Y, Months 7–12.
└── Eliminates lookahead bias; mirrors production deployment where detectors face future threats.
```

> **Recommendation:** Future studies should report performance across **both** Family-Disjoint and Chronological splits to guarantee real-world generalization.


## 6. Seven Critical Temporal Leakage Risks & Mitigations

| # | Temporal Leakage Risk | Failure Mechanism | Required Mitigation Protocol |
| :---: | :--- | :--- | :--- |
| **1** | **Same Execution Cross-Partition Leakage** | Windows from a single sample run split between train and test sets. | Partition strictly by `sample_id` before window slicing; never split windows from the same execution. |
| **2** | **Lookahead Feature Leakage** | Normalizing or computing rolling statistics using future windows $W_{k+1}, W_{k+2}$. | Use strictly causal windowing; rolling statistics must only access windows $\le k$. |
| **3** | **Global Preprocessing Leakage** | Fitting StandardScaler or PCA on the entire temporal corpus. | Fit all transformers strictly on the training partition; apply statically to test windows. |
| **4** | **Canary Artifact Timing Leakage** | Feature vectors directly recording the synthetic honeypot file paths. | Strip artificial canary file paths from feature tokens; use honeypot hits only for external ground truth labels. |
| **5** | **Shared Sandbox Environment Artifacts** | Artifacts left on disk from prior execution runs polluting subsequent runs. | Restore hypervisor to a clean snapshot before every execution session. |
| **6** | **Near-Duplicate Execution Traces** | Re-executing identical samples with minimal timing variation. | Verify uniqueness via behavioral trace hashing across distinct executions. |
| **7** | **Static Leakage in Sequential State** | Embedding full post-execution summary columns into intermediate observation windows. | Enforce that window $W_k$ contains only telemetry generated during $[t_0, t_k]$. |


In [12]:
fig_arch_path = figures_dir / 'temporal_memory_architecture.png'

fig, ax = plt.subplots(figsize=(10, 8.5), dpi=300)
ax.axis('off')

arch_nodes = [
    ('Isolated Controlled Lab Host', 'Isolated hypervisor host with zero external internet routing'),
    ('Virtual Machine Guest Sandbox', 'Clean guest OS snapshot with honeypot canary files pre-positioned'),
    ('Instrumented Target Process', 'Process monitored via kernel-level ETW and Minifilter driver'),
    ('Event Collector Subsystem', 'High-speed circular ring buffer: API, DLL, Registry, File, Process events'),
    ('Volatile Memory Snapshot Collector', 'Periodic VMI memory page inspection and entropy scanner'),
    ('Timestamp Synchronization Hub', 'Microsecond precision event alignment and UTC monotonic clock tagging'),
    ('Sliding Window Aggregator', 'Configurable windowing: W_k in [t_start, t_end), rate computation'),
    ('Temporal Feature Engineering Engine', 'Calculation of delta_entropy, delta_memory, rolling means'),
    ('Temporal Research Dataset Repository', 'Partitioned parquet/CSV store with ground-truth onset timestamps'),
    ('Multi-Paradigm ML & PPO Training', 'Sequential PPO agent and temporal Random Forest / XGBoost classifiers'),
]

y_positions = np.linspace(0.93, 0.07, len(arch_nodes))

for i, ((title, desc), y) in enumerate(zip(arch_nodes, y_positions)):
    color_box = '#e8f4f8' if i < 3 else ('#e8f8ec' if i < 6 else ('#fff8e7' if i < 9 else '#fde8e8'))
    color_edge = '#2b7bba' if i < 3 else ('#2bba52' if i < 6 else ('#ba852b' if i < 9 else '#ba2b2b'))
    
    props = dict(boxstyle='round,pad=0.55', facecolor=color_box, edgecolor=color_edge, linewidth=1.5)
    ax.text(0.5, y, f'{i+1}. {title}\n[{desc}]', ha='center', va='center', fontsize=9.2, fontweight='bold', bbox=props)
    
    if i < len(arch_nodes) - 1:
        next_y = y_positions[i+1]
        ax.annotate('', xy=(0.5, next_y + 0.024), xytext=(0.5, y - 0.024),
                    arrowprops=dict(arrowstyle='->', color='#444444', lw=1.5))

plt.title('Proposed Future Real Temporal & Memory Data Collection Architecture\n[Proposed Design Specification — Not Currently Operational]', fontsize=11.5, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig(fig_arch_path)
plt.show()
plt.close()

print(f'Saved temporal collection architecture diagram to: {fig_arch_path}')


Saved temporal collection architecture diagram to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\figures\temporal_memory_architecture.png


## 7. Safety, Ethics, and Experimental Containment Boundary

> [!CAUTION]
> **STRICT SAFETY DIRECTIVE & CONTAINMENT PROTOCOL:**  
> This project operates under strict defensive cybersecurity research guidelines.

### Mandatory Experimental Protocols for Future Implementation:
1. **Zero Malware Execution in Current Work:** This notebook and the surrounding research code contain **no executable malware, exploit payloads, or malicious scripts**.
2. **Air-Gapped Virtualization:** Any future empirical data collection must take place strictly inside isolated hypervisor guests (e.g. KVM/QEMU, VMware ESXi) on air-gapped hardware with all physical and virtual external network adapters disabled.
3. **No External Infrastructure Interaction:** No command-and-control (C2) domains, IP addresses, or external key servers may be contacted. Network traffic must terminate in simulated internal loopback responders (e.g. INetSim).
4. **Automated Snapshot Reversion:** Following every execution run, the guest operating system must be forcefully terminated and reverted to a pristine golden snapshot to prevent persistence or artifact contamination.
5. **Passive Sensor Telemetry:** Telemetry collection relies solely on native operating system logging facilities (ETW, Minifilter, hypervisor VMI) operating with administrative visibility.


In [13]:
print('Data Quality & Integrity Verification Suite:')
print('=' * 70)

def validate_temporal_dataset(df: pd.DataFrame, expected_schema: pd.DataFrame) -> Dict[str, Any]:
    """Execute programmatic data quality and integrity checks on a candidate temporal dataframe."""
    results = {'passed': True, 'checks': []}
    
    # Check 1: Required Columns Existence
    req_cols = expected_schema[expected_schema['Required?']]['Column'].tolist()
    missing = [c for c in req_cols if c not in df.columns]
    passed_1 = len(missing) == 0
    results['checks'].append(('Required Columns Present', passed_1, f'Missing: {missing}'))
    if not passed_1: results['passed'] = False
    
    # Check 2: Timestamp Monotonicity within Sample
    mono_pass = True
    if 'sample_id' in df.columns and 'timestamp_us' in df.columns:
        for s_id, group in df.groupby('sample_id'):
            if not group['timestamp_us'].is_monotonic_increasing:
                mono_pass = False
                break
    results['checks'].append(('Timestamps Monotonically Increasing', mono_pass, 'Monotonic check per sample'))
    if not mono_pass: results['passed'] = False
    
    # Check 3: Non-Negative Rates and Counts
    rate_cols = [c for c in df.columns if 'rate' in c or 'count' in c or 'bytes' in c]
    neg_pass = True
    if rate_cols:
        min_vals = df[rate_cols].min()
        if (min_vals < 0).any():
            neg_pass = False
    results['checks'].append(('No Negative Rates/Counts', neg_pass, 'Rate non-negativity constraint'))
    if not neg_pass: results['passed'] = False
    
    # Check 4: No Missing Values in Core Identification
    null_pass = True
    core_cols = [c for c in ['sample_id', 'timestamp_us', 'observation_window', 'label'] if c in df.columns]
    if core_cols and df[core_cols].isnull().any().any():
        null_pass = False
    results['checks'].append(('Zero Nulls in Core Metadata', null_pass, 'Primary key integrity'))
    if not null_pass: results['passed'] = False
    
    return results

print('Defined validate_temporal_dataset() function successfully.')


Data Quality & Integrity Verification Suite:
Defined validate_temporal_dataset() function successfully.


In [14]:
# Generate SMALL, ARTIFICIAL demonstration dataframe solely to test the validation functions
# WARNING: ARTIFICIAL SYNTHETIC DATA FOR CODE TESTING ONLY - NOT MALWARE DATA
np.random.seed(42)

demo_rows = []
for sample_num, s_id, lbl, fam in [('S1', 'sample-demo-001', 'benign', 'benign_baseline'), ('S2', 'sample-demo-002', 'ransomware', 'synthetic_family_G')]:
    base_time = 1727600000000000 # epoch us
    for w in range(5):
        t_us = base_time + w * 100000 # 100ms steps
        rel_ms = w * 100.0
        is_rw = (lbl == 'ransomware') and (w >= 2)
        demo_rows.append({
            'sample_id': s_id,
            'process_id': 1044 if sample_num == 'S1' else 4088,
            'parent_process_id': 600,
            'timestamp_us': t_us,
            'relative_time_ms': rel_ms,
            'observation_window': w,
            'process_name': 'benign_task.exe' if sample_num == 'S1' else 'worker_process.exe',
            'process_path': 'C:\\Windows\\System32\\benign_task.exe',
            'label': lbl,
            'family': fam,
            'api_count': 12 if not is_rw else 145,
            'api_rate': 0.12 if not is_rw else 1.45,
            'dll_count': 0 if w > 0 else 14,
            'dll_load_rate': 0.0,
            'registry_read_count': 5 if not is_rw else 34,
            'registry_write_count': 1 if not is_rw else 28,
            'file_read_count': 2 if not is_rw else 80,
            'file_write_count': 0 if not is_rw else 95,
            'file_create_count': 0 if not is_rw else 22,
            'directory_activity_count': 1 if not is_rw else 12,
            'entropy': 4.1 if not is_rw else 7.8,
            'memory_snapshot_id': f'snap-{s_id}-{w}',
            'working_set_bytes': int(18000000 + w * 250000),
            'private_bytes': int(12000000 + w * 400000),
            'handle_count': 120 + w * 2,
            'thread_count': 4 if not is_rw else 8,
            'network_connection_count': 1,
            'loaded_module_count': 18,
        })

df_demo = pd.DataFrame(demo_rows)
print('Generated synthetic validation demo dataframe (10 rows, 2 samples):')
print(f'Shape: {df_demo.shape}')
print('NOTE: Schema validation demo only — not research data.')


Generated synthetic validation demo dataframe (10 rows, 2 samples):
Shape: (10, 28)
NOTE: Schema validation demo only — not research data.


In [15]:
val_report = validate_temporal_dataset(df_demo, df_schema)

print('=' * 70)
print('VALIDATION SUITE TEST RESULTS ON DEMO DATA:')
print('=' * 70)
print(f'Overall Integrity Status: {"PASSED (ALL CHECKS OK)" if val_report["passed"] else "FAILED"}\n')

for name, passed, details in val_report['checks']:
    status = 'PASS' if passed else 'FAIL'
    print(f'  [{status:<4}] {name:<35} : {details}')

assert val_report['passed'], 'Validation suite failed on valid demo data!'


VALIDATION SUITE TEST RESULTS ON DEMO DATA:
Overall Integrity Status: PASSED (ALL CHECKS OK)

  [PASS] Required Columns Present            : Missing: []
  [PASS] Timestamps Monotonically Increasing : Monotonic check per sample
  [PASS] No Negative Rates/Counts            : Rate non-negativity constraint
  [PASS] Zero Nulls in Core Metadata         : Primary key integrity


In [16]:
print('Illustrative Row-Level Future Telemetry Stream (Sample Extract):')
print('=' * 70)
display_cols = [
    'sample_id', 'timestamp_us', 'observation_window', 'api_rate',
    'file_write_count', 'entropy', 'private_bytes', 'handle_count', 'label'
]
print(df_demo[display_cols].to_string(index=False))
print('\nLabel: Illustrative schema — not collected data.')


Illustrative Row-Level Future Telemetry Stream (Sample Extract):
      sample_id     timestamp_us  observation_window  api_rate  file_write_count  entropy  private_bytes  handle_count      label
sample-demo-001 1727600000000000                   0      0.12                 0      4.1       12000000           120     benign
sample-demo-001 1727600000100000                   1      0.12                 0      4.1       12400000           122     benign
sample-demo-001 1727600000200000                   2      0.12                 0      4.1       12800000           124     benign
sample-demo-001 1727600000300000                   3      0.12                 0      4.1       13200000           126     benign
sample-demo-001 1727600000400000                   4      0.12                 0      4.1       13600000           128     benign
sample-demo-002 1727600000000000                   0      0.12                 0      4.1       12000000           120 ransomware
sample-demo-002 172760000

In [17]:
metrics_feasibility = pd.DataFrame([
    {'Metric': 'Classification Accuracy', 'Requires Temporal Data?': 'No', 'Currently Available?': 'Yes (Notebooks 02, 03, 06)', 'Scientific Scope': 'One-shot static taxonomy discrimination'},
    {'Metric': 'Balanced Accuracy / Macro F1', 'Requires Temporal Data?': 'No', 'Currently Available?': 'Yes (Notebooks 02, 03, 06)', 'Scientific Scope': 'Static class-weighted evaluation'},
    {'Metric': 'False Positive Rate', 'Requires Temporal Data?': 'Yes (over time)', 'Currently Available?': 'Yes (Static proxy)', 'Scientific Scope': 'Rate of false alarms on benign processes'},
    {'Metric': 'Physical Detection Latency', 'Requires Temporal Data?': 'Yes', 'Currently Available?': 'No (Future Only)', 'Scientific Scope': 'Time elapsed from onset (t_onset) to alert (t_alert)'},
    {'Metric': 'Time-to-Detection (TTD in ms)', 'Requires Temporal Data?': 'Yes', 'Currently Available?': 'No (Future Only)', 'Scientific Scope': 'Wall-clock milliseconds from process launch to quarantine'},
    {'Metric': 'Early Detection Rate (EDR)', 'Requires Temporal Data?': 'Yes', 'Currently Available?': 'No (Future Only)', 'Scientific Scope': 'Proportion of alerts issued before irreversible file damage'},
    {'Metric': 'Detection Before Encryption (DBE)', 'Requires Temporal Data?': 'Yes', 'Currently Available?': 'No (Future Only)', 'Scientific Scope': 'Binary verification that alert occurred prior to canary file modification'},
    {'Metric': 'Synthetic Alert Step', 'Requires Temporal Data?': 'No', 'Currently Available?': 'Yes (Notebook 05)', 'Scientific Scope': 'Observation step in synthetic proxy environment (t in [0, 4])'},
    {'Metric': 'Temporal Generalization', 'Requires Temporal Data?': 'Yes', 'Currently Available?': 'No (Future Only)', 'Scientific Scope': 'Evaluation across historical train vs. future test partitions'},
])

print('Research Metrics Feasibility Matrix (Current vs. Future Telemetry):')
print(metrics_feasibility[['Metric', 'Requires Temporal Data?', 'Currently Available?']].to_string(index=False))


Research Metrics Feasibility Matrix (Current vs. Future Telemetry):
                           Metric Requires Temporal Data?       Currently Available?
          Classification Accuracy                      No Yes (Notebooks 02, 03, 06)
     Balanced Accuracy / Macro F1                      No Yes (Notebooks 02, 03, 06)
              False Positive Rate         Yes (over time)         Yes (Static proxy)
       Physical Detection Latency                     Yes           No (Future Only)
    Time-to-Detection (TTD in ms)                     Yes           No (Future Only)
       Early Detection Rate (EDR)                     Yes           No (Future Only)
Detection Before Encryption (DBE)                     Yes           No (Future Only)
             Synthetic Alert Step                      No          Yes (Notebook 05)
          Temporal Generalization                     Yes           No (Future Only)


## 8. Migration Plan: Synthetic Proxy to Streaming Environment

To transition from the current research state to the future real temporal system without invalidating previous codebase architecture, the software transition follows a modular subclassing design:

```
gymnasium.Env
      │
      ▼
RansomwareDetectionEnv (src/ppo_environment.py)
[Current Synthetic Proxy: reveals 16 static features across 5 stages]
      │
      │ Subclassed & Extended
      ▼
StreamingRansomwareDetectionEnv (Future src/streaming_ppo_env.py)
[Future Real Temporal Environment: ingests true timestamped sliding windows]
```

### Architectural Preservation Invariants:
1. **Identical Action Interface:** Discrete(3) — `{0: CONTINUE, 1: SUSPICIOUS, 2: ALERT}` remains unchanged.
2. **Gymnasium Compliance:** Adheres to standard `reset(options)` and `step(action) -> obs, reward, term, trunc, info` signatures.
3. **PPO Model Compatibility:** The same Stable-Baselines3 PPO architecture (`MlpPolicy`) trained in `notebooks/05_train_ppo.ipynb` can directly ingest real temporal observations once input dimensions are matched.


In [18]:
roadmap_stages = pd.DataFrame([
    {'Stage': 'Stage 1', 'Focus Area': 'Schema Specification', 'Deliverable': 'Validated row-level schema & window specifications (This Notebook)'},
    {'Stage': 'Stage 2', 'Focus Area': 'Safe Lab Infrastructure', 'Deliverable': 'Air-gapped hypervisor testbed with golden snapshots & canary honeypots'},
    {'Stage': 'Stage 3', 'Focus Area': 'Timestamp Synchronization', 'Deliverable': 'High-resolution ETW & kernel hook pipeline with microsecond UTC clock'},
    {'Stage': 'Stage 4', 'Focus Area': 'Memory Feature Extraction', 'Deliverable': 'Hypervisor VMI memory page entropy scanner and allocation logger'},
    {'Stage': 'Stage 5', 'Focus Area': 'Temporal Window Aggregator', 'Deliverable': 'Sliding window buffer engine producing chronological W_k vectors'},
    {'Stage': 'Stage 6', 'Focus Area': 'Leakage-Controlled Splitting', 'Deliverable': 'Sample-disjoint and family-disjoint train/test partition pipelines'},
    {'Stage': 'Stage 7', 'Focus Area': 'Temporal Supervised Baselines', 'Deliverable': 'Random Forest & XGBoost window-level early classification models'},
    {'Stage': 'Stage 8', 'Focus Area': 'Real Temporal PPO Agent', 'Deliverable': 'Streaming PPO training against real progressive behavioral sequences'},
    {'Stage': 'Stage 9', 'Focus Area': 'Physical Latency Evaluation', 'Deliverable': 'Empirical measurement of time-to-detection and early detection rate'},
    {'Stage': 'Stage 10', 'Focus Area': 'Independent Generalization', 'Deliverable': 'Validation on novel zero-day malware families and external sandbox logs'},
])

print('Ten-Stage Research Implementation Roadmap:')
print(roadmap_stages.to_string(index=False))


Ten-Stage Research Implementation Roadmap:
   Stage                    Focus Area                                                             Deliverable
 Stage 1          Schema Specification      Validated row-level schema & window specifications (This Notebook)
 Stage 2       Safe Lab Infrastructure  Air-gapped hypervisor testbed with golden snapshots & canary honeypots
 Stage 3     Timestamp Synchronization   High-resolution ETW & kernel hook pipeline with microsecond UTC clock
 Stage 4     Memory Feature Extraction        Hypervisor VMI memory page entropy scanner and allocation logger
 Stage 5    Temporal Window Aggregator        Sliding window buffer engine producing chronological W_k vectors
 Stage 6  Leakage-Controlled Splitting      Sample-disjoint and family-disjoint train/test partition pipelines
 Stage 7 Temporal Supervised Baselines        Random Forest & XGBoost window-level early classification models
 Stage 8       Real Temporal PPO Agent    Streaming PPO training agai

## 9. Final Research Limitations & Scientific Guardrails

### Explicit Methodological Boundaries:
- **Design Specification Only:** This notebook provides a forward-looking architectural and schema design for future empirical collection. It does **NOT** collect real ransomware data.
- **No Malware Execution:** No malware was executed, analyzed dynamically, or downloaded to create these specifications.
- **No Physical Latency Claims:** Real-time detection latency (in milliseconds or seconds) has **NOT** been demonstrated in this project to date.
- **Synthetic Proxy Preservation:** The PPO results obtained in Notebook 05 and compared in Notebook 06 remain a **synthetic temporal proof-of-concept**. They illustrate sequential decision mechanics under simulated progressive information gain, but cannot be presented as real-time endpoint performance until the Stage 2–10 roadmap is empirically executed.


In [19]:
final_report_path = reports_dir / 'temporal_memory_data_design.md'

report_markdown = r"""# Real Temporal and Memory Data Collection Design Specification

**Project:** Early-Stage Ransomware Detection via Dynamic Memory Analysis and Reinforcement Learning  
**Document:** Research Specification for Future Temporal & Memory Telemetry Pipeline  
**Date:** September 2026  
**Status:** Design and Schema Phase Completed  

---

## 1. Research Gap
The baseline machine learning models (Random Forest, XGBoost) and reinforcement learning agent (PPO) were developed using a static/dynamic feature-summary dataset (`cleaned_dataset.csv`). While achieving 100% macro F1 on complete feature vectors, this dataset lacks physical execution timestamps, event sequence ordering, and dynamic memory page telemetry. Consequently, true physical early-stage detection (latency in milliseconds, detection prior to initial encryption) cannot currently be measured.

---

## 2. Current Dataset
- **Nature:** Post-mortem dynamic-analysis aggregate feature counts and entropy summaries.
- **Rows:** 1,750 samples across 3 ransomware families (G, E, L).
- **Evaluation:** Single-shot static evaluation or synthetic progressive feature revelation (5 steps: 20% to 100%).

---

## 3. Proposed Future Dataset
A high-resolution, continuous streaming event dataset capturing discrete sliding observation windows ($W_k$) from isolated hypervisor execution runs, recording event rates, memory deltas, and microsecond timestamps.

---

## 4. Temporal Schema
The proposed dataset schema comprises 27 structured fields spanning session identifiers, process tree metadata, microsecond timestamps, and behavioral rate counters (API calls/ms, file creations/s, registry modifications/s).

---

## 5. Memory Schema
Volatile memory telemetry captures working set byte size, private committed bytes, unbacked executable memory regions, heap allocation entropy, and handle/thread counts.

---

## 6. Feature Engineering
Raw circular event streams are transformed into temporal window observations via first-order deltas ($\Delta \text{entropy}$, $\Delta \text{memory}$, $\Delta \text{handles}$) and moving average rolling statistics.

---

## 7. Labeling
Dual-level labeling architecture: sample-level ground truth (benign vs. ransomware family) and window-level temporal phase labels (pre-encryption suspicious, active encryption, post-damage).

---

## 8. Ground Truth
Hardware-synchronized milestone logging tracking process creation, code injection, volume shadow tampering, and pre-positioned canary file access ($t_{\text{onset}}$).

---

## 9. Leakage Prevention
Strict prevention protocols addressing 7 temporal leakage vectors, including sample-disjoint partitioning, causal rolling window calculation, and golden hypervisor snapshot reversion.

---

## 10. PPO Integration
Smooth architectural migration from `RansomwareDetectionEnv` to `StreamingRansomwareDetectionEnv`, preserving the Discrete(3) action space (`CONTINUE`, `SUSPICIOUS`, `ALERT`) and Gymnasium API contracts.

---

## 11. Evaluation Metrics
Enables future physical metrics: Time-to-Detection (TTD in ms), Early Detection Rate (EDR), and Detection Before First Encryption (DBE).

---

## 12. Safety Boundary
All empirical data collection is strictly restricted to air-gapped, isolated hypervisor testbeds with automated snapshot rollbacks. Zero malware execution is performed in this design phase.

---

## 13. Implementation Roadmap
A 10-stage sequential development roadmap taking the project from schema specification to independent multi-sandbox zero-day validation.

---

## 14. Limitations
This specification designs future collection infrastructure; it does not collect real malware data or establish real-time latency on the current dataset. PPO results remain a synthetic proof-of-concept until real temporal telemetry is collected."""

with open(final_report_path, 'w', encoding='utf-8') as f:
    f.write(report_markdown)

print(f'Successfully wrote comprehensive final design report to: {final_report_path}')
print('\nTemporal and memory data design completed. The current PPO results remain a synthetic proof-of-concept until real timestamped observations are collected.')


Successfully wrote comprehensive final design report to: C:\Users\devar\OneDrive\Documents\Ransomware\ransomware\results\reports\temporal_memory_data_design.md

Temporal and memory data design completed. The current PPO results remain a synthetic proof-of-concept until real timestamped observations are collected.
